In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/ishanmanglik/div2k/tensor_x.npy
/kaggle/input/notebooks/ishanmanglik/div2k/__results__.html
/kaggle/input/notebooks/ishanmanglik/div2k/val_tensor_x.npy
/kaggle/input/notebooks/ishanmanglik/div2k/__notebook__.ipynb
/kaggle/input/notebooks/ishanmanglik/div2k/__output__.json
/kaggle/input/notebooks/ishanmanglik/div2k/val_tensor_y.npy
/kaggle/input/notebooks/ishanmanglik/div2k/tensor_y.npy
/kaggle/input/notebooks/ishanmanglik/div2k/custom.css
/kaggle/input/notebooks/ishanmanglik/div2k/__results___files/__results___29_1.png
/kaggle/input/notebooks/ishanmanglik/div2k/__results___files/__results___26_3.png
/kaggle/input/notebooks/ishanmanglik/div2k/__results___files/__results___26_2.png
/kaggle/input/notebooks/ishanmanglik/div2k/__results___files/__results___12_1.png
/kaggle/input/notebooks/ishanmanglik/div2k/__results___files/__results___26_4.png
/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_valid_HR/DIV2K_valid_HR/0857.png
/kaggle/input/data

# ESRT — Single Image Super-Resolution: Small / Medium / Default Sweep

Self-contained notebook: patched ESRT model + DIV2K dataset + training loop
+ evaluation, for all 3 size configurations in sequence.

**Patched from the official ESRT implementation:**
1. `n_feats` and `n_blocks` exposed as constructor args (were hardcoded).
2. Attention `dim` fixed to scale with `n_feats` (was hardcoded to 288,
   which only matched the original default n_feats=32).

**Verified working sizes:** n_feats must be a multiple of 16 (hardcoded
`num_heads=8` inside the attention block requires `(n_feats*9)//2 % 8 == 0`).
Configs below use 16 / 32 / 48.


Setup & Imports

In [53]:
import os
import math
import time
import json
import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
LOG_DIR = "/kaggle/working/logs"
RESULTS_DIR = "/kaggle/working/results"
for d in [CHECKPOINT_DIR, LOG_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

Device: cuda


## 2. Model: `common.py` components

Shared building blocks ESRT depends on (unchanged from the official repo).


In [54]:
class DefaultConv:
    """Factory matching the original common.default_conv signature."""
    def __call__(self, in_channels, out_channels, kernel_size, bias=True, groups=1):
        return nn.Conv2d(
            in_channels, out_channels, kernel_size,
            padding=(kernel_size // 2), bias=bias, groups=groups)

default_conv = DefaultConv()


class Scale(nn.Module):
    def __init__(self, init_value=1e-3):
        super().__init__()
        self.scale = nn.Parameter(torch.FloatTensor([init_value]))

    def forward(self, input):
        return input * self.scale


class Upsampler(nn.Sequential):
    def __init__(self, conv, scale, n_feats, bn=False, act=False, bias=True):
        m = []
        if (scale & (scale - 1)) == 0:  # scale = 2^n
            for _ in range(int(math.log(scale, 2))):
                m.append(conv(n_feats, 4 * n_feats, 3, bias))
                m.append(nn.PixelShuffle(2))
                if bn:
                    m.append(nn.BatchNorm2d(n_feats))
                if act == "relu":
                    m.append(nn.ReLU(True))
                elif act == "prelu":
                    m.append(nn.PReLU(n_feats))
        elif scale == 3:
            m.append(conv(n_feats, 9 * n_feats, 3, bias))
            m.append(nn.PixelShuffle(3))
            if bn:
                m.append(nn.BatchNorm2d(n_feats))
            if act == "relu":
                m.append(nn.ReLU(True))
            elif act == "prelu":
                m.append(nn.PReLU(n_feats))
        else:
            raise NotImplementedError
        super(Upsampler, self).__init__(*m)

## 3. Model: patch/attention utilities

`extract_image_patches` / `reverse_patches` (from `util/tools.py`) — used by
the MLABlock attention module to operate on local patches.


In [55]:
def same_padding(images, ksizes, strides, rates):
    assert len(images.size()) == 4
    batch_size, channel, rows, cols = images.size()
    out_rows = (rows + strides[0] - 1) // strides[0]
    out_cols = (cols + strides[1] - 1) // strides[1]
    effective_k_row = (ksizes[0] - 1) * rates[0] + 1
    effective_k_col = (ksizes[1] - 1) * rates[1] + 1
    padding_rows = max(0, (out_rows - 1) * strides[0] + effective_k_row - rows)
    padding_cols = max(0, (out_cols - 1) * strides[1] + effective_k_col - cols)
    padding_top = int(padding_rows / 2.0)
    padding_left = int(padding_cols / 2.0)
    padding_bottom = padding_rows - padding_top
    padding_right = padding_cols - padding_left
    paddings = (padding_left, padding_right, padding_top, padding_bottom)
    images = nn.ZeroPad2d(paddings)(images)
    return images


def extract_image_patches(images, ksizes, strides, rates, padding="same"):
    assert len(images.size()) == 4
    assert padding in ["same", "valid"]
    if padding == "same":
        images = same_padding(images, ksizes, strides, rates)
    unfold = nn.Unfold(kernel_size=ksizes, dilation=rates, padding=0, stride=strides)
    return unfold(images)


def reverse_patches(images, out_size, ksizes, strides, padding):
    fold = nn.Fold(output_size=out_size, kernel_size=ksizes, dilation=1,
                    padding=padding, stride=strides)
    return fold(images)


## 4. Model: transformer attention block (`MLABlock`)

In [56]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    png_files = [f for f in files if f.endswith('.png')]
    if png_files:
        print(root, '->', len(png_files), 'png files, e.g.', png_files[0])

/kaggle/input/notebooks/ishanmanglik/div2k/__results___files -> 5 png files, e.g. __results___29_1.png
/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_valid_HR/DIV2K_valid_HR -> 100 png files, e.g. 0857.png
/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR -> 800 png files, e.g. 0566.png


In [57]:
  class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features // 4
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class EffAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None,
                 attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.reduce = nn.Linear(dim, dim // 2, bias=qkv_bias)
        self.qkv = nn.Linear(dim // 2, dim // 2 * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim // 2, dim)
        self.attn_drop = nn.Dropout(attn_drop)

    def forward(self, x):
        x = self.reduce(x)
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q_all = torch.split(q, math.ceil(N // 4), dim=-2)
        k_all = torch.split(k, math.ceil(N // 4), dim=-2)
        v_all = torch.split(v, math.ceil(N // 4), dim=-2)

        output = []
        for qi, ki, vi in zip(q_all, k_all, v_all):
            attn = (qi @ ki.transpose(-2, -1)) * self.scale
            attn = attn.softmax(dim=-1)
            attn = self.attn_drop(attn)
            trans_x = (attn @ vi).transpose(1, 2)
            output.append(trans_x)

        x = torch.cat(output, dim=1)
        x = x.reshape(B, N, C)
        x = self.proj(x)
        return x


class MLABlock(nn.Module):
    """
    dim must equal n_feats * 9 (3x3 patch extraction over n_feats channels).
    Official repo hardcoded dim=288, which only matches n_feats=32.
    PATCHED: dim is now derived from n_feats at the call site.
    """
    def __init__(self, n_feat=64, dim=768, num_heads=8, mlp_ratio=4.0,
                 qkv_bias=False, qk_scale=None, drop=0.0, attn_drop=0.0,
                 drop_path=0.0, act_layer=nn.ReLU, norm_layer=nn.LayerNorm):
        super(MLABlock, self).__init__()
        self.dim = dim
        self.atten = EffAttention(self.dim, num_heads=8, qkv_bias=False,
                                   qk_scale=None, attn_drop=0.0, proj_drop=0.0)
        self.norm1 = nn.LayerNorm(self.dim)
        self.mlp = Mlp(in_features=dim, hidden_features=dim // 4,
                        act_layer=act_layer, drop=drop)
        self.norm2 = nn.LayerNorm(self.dim)

    def forward(self, x):
        x = extract_image_patches(x, ksizes=[3, 3], strides=[1, 1],
                                   rates=[1, 1], padding="same")
        x = x.permute(0, 2, 1)
        x = x + self.atten(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## 5. Model: ESRT body blocks (`CALayer`, `one_module`, `Updownblock`, `Un`)


In [58]:
class CALayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(CALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_du = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, 1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, channel, 1, padding=0, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv_du(y)
        return x * y


class one_conv(nn.Module):
    def __init__(self, inchanels, growth_rate, kernel_size=3, relu=True):
        super(one_conv, self).__init__()
        self.conv = nn.Conv2d(inchanels, growth_rate, kernel_size=kernel_size,
                               padding=kernel_size >> 1, stride=1)
        self.flag = relu
        self.conv1 = nn.Conv2d(growth_rate, inchanels, kernel_size=kernel_size,
                                padding=kernel_size >> 1, stride=1)
        if relu:
            self.relu = nn.PReLU(growth_rate)
        self.weight1 = Scale(1)
        self.weight2 = Scale(1)

    def forward(self, x):
        if self.flag == False:
            output = self.weight1(x) + self.weight2(self.conv1(self.conv(x)))
        else:
            output = self.weight1(x) + self.weight2(self.conv1(self.relu(self.conv(x))))
        return output


class BasicConv(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=1,
                 dilation=1, groups=1, relu=True, bn=False, bias=False,
                 up_size=0, fan=False):
        super(BasicConv, self).__init__()
        self.out_channels = out_planes
        self.in_channels = in_planes
        if fan:
            self.conv = nn.ConvTranspose2d(in_planes, out_planes, kernel_size=kernel_size,
                                            stride=stride, padding=padding,
                                            dilation=dilation, groups=groups, bias=bias)
        else:
            self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                                   stride=stride, padding=padding,
                                   dilation=dilation, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_planes, eps=1e-5, momentum=0.01, affine=True) if bn else None
        self.relu = nn.ReLU(inplace=True) if relu else None
        self.up_size = up_size
        self.up_sample = nn.Upsample(size=(up_size, up_size), mode="bilinear") if up_size != 0 else None

    def forward(self, x):
        x = self.conv(x)
        if self.bn is not None:
            x = self.bn(x)
        if self.relu is not None:
            x = self.relu(x)
        if self.up_size > 0:
            x = self.up_sample(x)
        return x


class one_module(nn.Module):
    def __init__(self, n_feats):
        super(one_module, self).__init__()
        self.layer1 = one_conv(n_feats, n_feats // 2, 3)
        self.layer2 = one_conv(n_feats, n_feats // 2, 3)
        self.layer4 = BasicConv(n_feats, n_feats, 3, 1, 1)
        self.alise = BasicConv(2 * n_feats, n_feats, 1, 1, 0)
        self.atten = CALayer(n_feats)
        self.weight1 = Scale(1)
        self.weight2 = Scale(1)
        self.weight3 = Scale(1)
        self.weight4 = Scale(1)
        self.weight5 = Scale(1)

    def forward(self, x):
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x4 = self.layer4(self.atten(self.alise(torch.cat([self.weight2(x2), self.weight3(x1)], 1))))
        return self.weight4(x) + self.weight5(x4)


class Updownblock(nn.Module):
    def __init__(self, n_feats):
        super(Updownblock, self).__init__()
        self.encoder = one_module(n_feats)
        self.decoder_low = one_module(n_feats)
        self.decoder_high = one_module(n_feats)
        self.alise = one_module(n_feats)
        self.alise2 = BasicConv(2 * n_feats, n_feats, 1, 1, 0)
        self.down = nn.AvgPool2d(kernel_size=2)
        self.att = CALayer(n_feats)

    def forward(self, x):
        x1 = self.encoder(x)
        x2 = self.down(x1)
        high = x1 - F.interpolate(x2, size=x.size()[-2:], mode="bilinear", align_corners=True)
        for i in range(5):
            x2 = self.decoder_low(x2)
        x3 = x2
        high1 = self.decoder_high(high)
        x4 = F.interpolate(x3, size=x.size()[-2:], mode="bilinear", align_corners=True)
        return self.alise(self.att(self.alise2(torch.cat([x4, high1], dim=1)))) + x


class Un(nn.Module):
    """PATCHED: attention dim now derived as n_feats * 9 instead of hardcoded 288."""
    def __init__(self, n_feats, wn):
        super(Un, self).__init__()
        self.encoder1 = Updownblock(n_feats)
        self.encoder2 = Updownblock(n_feats)
        self.encoder3 = Updownblock(n_feats)
        self.reduce = default_conv(3 * n_feats, n_feats, 3)
        self.weight2 = Scale(1)
        self.weight1 = Scale(1)
        self.attention = MLABlock(n_feat=n_feats, dim=n_feats * 9)  # PATCHED
        self.alise = default_conv(n_feats, n_feats, 3)

    def forward(self, x):
        x1 = self.encoder1(x)
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        b, c, h, w = x3.shape
        out = self.attention(self.reduce(torch.cat([x1, x2, x3], dim=1)))
        out = out.permute(0, 2, 1)
        out = reverse_patches(out, (h, w), (3, 3), 1, 1)
        out = self.alise(out)
        return self.weight1(x) + self.weight2(out)



## 6. Model: `ESRT` (top-level)

**PATCHED:** `n_feats` and `n_blocks` are now constructor arguments
(originally hardcoded to 32 and 1 respectively). Defaults match the
original paper.

**Verified constraint:** `n_feats` must be a multiple of 16, or the
attention block's internal reshape fails (hardcoded `num_heads=8`).


In [59]:
class ESRT(nn.Module):
    def __init__(self, upscale=4, conv=default_conv, n_feats=32, n_blocks=1):
        super(ESRT, self).__init__()
        wn = lambda x: torch.nn.utils.weight_norm(x)
        kernel_size = 3
        scale = upscale
        self.n_blocks = n_blocks

        modules_head = [conv(3, n_feats, kernel_size)]

        modules_body = nn.ModuleList()
        for i in range(n_blocks):
            modules_body.append(Un(n_feats=n_feats, wn=wn))

        modules_tail = [
            Upsampler(conv, scale, n_feats, act=False),
            conv(n_feats, 3, kernel_size),
        ]

        self.up = nn.Sequential(
            Upsampler(conv, scale, n_feats, act=False),
            BasicConv(n_feats, 3, 3, 1, 1),
        )
        self.head = nn.Sequential(*modules_head)
        self.body = nn.Sequential(*modules_body)
        self.tail = nn.Sequential(*modules_tail)
        self.reduce = conv(n_blocks * n_feats, n_feats, kernel_size)

    def forward(self, x1, x2=None, test=False):
        x1 = self.head(x1)
        res2 = x1
        body_out = []
        for i in range(self.n_blocks):
            x1 = self.body[i](x1)
            body_out.append(x1)
        res1 = torch.cat(body_out, 1)
        res1 = self.reduce(res1)

        x1 = self.tail(res1)
        x1 = self.up(res2) + x1
        return x1


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6


# Quick sanity check across all 3 sizes before training anything
print("Sanity check (forward pass on dummy input):")
for name, kwargs in [
    ("esrt_small",   dict(upscale=4, n_feats=16, n_blocks=1)),
    ("esrt_medium",  dict(upscale=4, n_feats=32, n_blocks=1)),
    ("esrt_default", dict(upscale=4, n_feats=48, n_blocks=1)),
]:
    m = ESRT(**kwargs)
    out = m(torch.randn(1, 3, 48, 48))
    print(f"  {name}: params={count_params(m):.3f}M, output shape={tuple(out.shape)}")
del m, out


Sanity check (forward pass on dummy input):
  esrt_small: params=0.190M, output shape=(1, 3, 192, 192)
  esrt_medium: params=0.752M, output shape=(1, 3, 192, 192)
  esrt_default: params=1.686M, output shape=(1, 3, 192, 192)


## 7. Configs

Three size tiers. `n_blocks` kept at 1 for all — only width (`n_feats`)
is swept, since depth scaling (`n_blocks > 1`) hasn't been verified and
risks much longer training time.


In [60]:
ESRT_CONFIGS = {
    "esrt_small":   dict(upscale=4, n_feats=16, n_blocks=1),   # ~0.19M params
    "esrt_medium":  dict(upscale=4, n_feats=32, n_blocks=1),   # ~0.75M params (paper default)
    "esrt_default": dict(upscale=4, n_feats=48, n_blocks=1),   # ~1.69M params
}


In [61]:
class SRDataset(Dataset):
    def __init__(self, hr_folder, scale=4, hr_size=256, n_images=None):
        self.hr_paths = sorted(glob.glob(os.path.join(hr_folder, "*.png")))
        if n_images is not None:
            self.hr_paths = self.hr_paths[:n_images]
        self.scale = scale
        self.hr_transform = transforms.Compose([
            transforms.Resize((hr_size, hr_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr = Image.open(self.hr_paths[idx]).convert("RGB")
        hr = self.hr_transform(hr)
        lr = transforms.functional.resize(
            hr,
            [hr.shape[-2] // self.scale, hr.shape[-1] // self.scale],
            interpolation=transforms.InterpolationMode.BICUBIC,
        )
        return lr, hr


# --- EDIT THIS PATH to match your dataset location ---
HR_FOLDER = "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR"
VAL_HR_FOLDER = HR_FOLDER

SCALE = 4
HR_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 2

full_dataset = SRDataset(HR_FOLDER, scale=SCALE, hr_size=HR_SIZE)
n_val = 20
n_train = len(full_dataset) - n_val
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")

lr_sample, hr_sample = next(iter(train_loader))
print("LR batch shape:", lr_sample.shape, "| HR batch shape:", hr_sample.shape)

Train images: 780 | Val images: 20
LR batch shape: torch.Size([4, 3, 64, 64]) | HR batch shape: torch.Size([4, 3, 256, 256])


In [62]:
def train_one_config(model_class, config, run_name, train_loader, device,
                      num_epochs=20, lr=2e-4, checkpoint_every=5):
    model = model_class(**config).to(device)
    params_m = count_params(model)
    print(f"[{run_name}] Params: {params_m:.3f} M")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    epoch_losses = []
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for lr_img, hr_img in train_loader:
            lr_img, hr_img = lr_img.to(device), hr_img.to(device)

            optimizer.zero_grad()
            sr_img = model(lr_img)
            loss = criterion(sr_img, hr_img)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        epoch_losses.append(avg_loss)
        print(f"[{run_name}] Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

        if (epoch + 1) % checkpoint_every == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"{run_name}_epoch{epoch+1}.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(f"[{run_name}] Checkpoint saved: {ckpt_path}")

    total_time = time.time() - start_time

    final_ckpt = os.path.join(CHECKPOINT_DIR, f"{run_name}_final.pth")
    torch.save(model.state_dict(), final_ckpt)

    log = {
        "run_name": run_name,
        "config": config,
        "params_M": params_m,
        "num_epochs": num_epochs,
        "epoch_losses": epoch_losses,
        "final_loss": epoch_losses[-1],
        "total_train_time_sec": total_time,
        "avg_time_per_epoch_sec": total_time / num_epochs,
        "checkpoint_path": final_ckpt,
    }

    log_path = os.path.join(LOG_DIR, f"{run_name}_log.json")
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)

    print(f"[{run_name}] Done. Total time: {total_time/60:.1f} min. Log: {log_path}")
    return log, model

In [63]:
def evaluate_model(model, val_loader, device, name="model", warmup=2):
    model.eval()

    psnr_scores, ssim_scores = [], []
    psnr_bic, ssim_bic = [], []
    inference_times = []

    with torch.no_grad():
        for i, (lr_img, hr_img) in enumerate(val_loader):
            lr_img, hr_img = lr_img.to(device), hr_img.to(device)

            if i < warmup:
                _ = model(lr_img)
                if device.type == "cuda":
                    torch.cuda.synchronize()

            start = time.time()
            sr = model(lr_img)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.time() - start
            inference_times.append(elapsed)

            sr = torch.clamp(sr, 0, 1)
            bic = torch.clamp(
                F.interpolate(lr_img, scale_factor=SCALE, mode="bicubic", align_corners=False), 0, 1
            )

            sr_np = sr.squeeze().cpu().numpy().transpose(1, 2, 0)
            bic_np = bic.squeeze().cpu().numpy().transpose(1, 2, 0)
            hr_np = hr_img.squeeze().cpu().numpy().transpose(1, 2, 0)

            psnr_scores.append(psnr(hr_np, sr_np, data_range=1.0))
            ssim_scores.append(ssim(hr_np, sr_np, channel_axis=2, data_range=1.0))
            psnr_bic.append(psnr(hr_np, bic_np, data_range=1.0))
            ssim_bic.append(ssim(hr_np, bic_np, channel_axis=2, data_range=1.0))

    results = {
        "name": name,
        "psnr": float(np.mean(psnr_scores)),
        "ssim": float(np.mean(ssim_scores)),
        "psnr_bicubic": float(np.mean(psnr_bic)),
        "ssim_bicubic": float(np.mean(ssim_bic)),
        "params_M": count_params(model),
        "avg_inference_ms": float(np.mean(inference_times) * 1000),
    }

    print("=" * 55)
    print(f"Results: {name}")
    print("=" * 55)
    print(f"PSNR: {results['psnr']:.2f} dB   (bicubic: {results['psnr_bicubic']:.2f} dB)")
    print(f"SSIM: {results['ssim']:.4f}      (bicubic: {results['ssim_bicubic']:.4f})")
    print(f"Params: {results['params_M']:.3f} M")
    print(f"Avg inference time: {results['avg_inference_ms']:.2f} ms/image")
    print("=" * 55)

    return results

In [64]:
NUM_EPOCHS = 20

all_logs = {}
all_results = {}

for run_name, config in ESRT_CONFIGS.items():
    print(f"\n{'='*60}\nStarting: {run_name}  config={config}\n{'='*60}")

    log, trained_model = train_one_config(
        model_class=ESRT,
        config=config,
        run_name=run_name,
        train_loader=train_loader,
        device=device,
        num_epochs=NUM_EPOCHS,
    )
    all_logs[run_name] = log

    result = evaluate_model(trained_model, val_loader, device, name=run_name)
    result["avg_time_per_epoch_sec"] = log["avg_time_per_epoch_sec"]
    result["total_train_time_sec"] = log["total_train_time_sec"]
    all_results[run_name] = result

    del trained_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\n\nAll ESRT runs complete.")


Starting: esrt_small  config={'upscale': 4, 'n_feats': 16, 'n_blocks': 1}
[esrt_small] Params: 0.190 M
[esrt_small] Epoch 1/20 - Loss: 0.1208
[esrt_small] Epoch 2/20 - Loss: 0.0700
[esrt_small] Epoch 3/20 - Loss: 0.0590
[esrt_small] Epoch 4/20 - Loss: 0.0534
[esrt_small] Epoch 5/20 - Loss: 0.0502
[esrt_small] Checkpoint saved: /kaggle/working/checkpoints/esrt_small_epoch5.pth
[esrt_small] Epoch 6/20 - Loss: 0.0482
[esrt_small] Epoch 7/20 - Loss: 0.0479
[esrt_small] Epoch 8/20 - Loss: 0.0467
[esrt_small] Epoch 9/20 - Loss: 0.0458
[esrt_small] Epoch 10/20 - Loss: 0.0452
[esrt_small] Checkpoint saved: /kaggle/working/checkpoints/esrt_small_epoch10.pth
[esrt_small] Epoch 11/20 - Loss: 0.0449
[esrt_small] Epoch 12/20 - Loss: 0.0451
[esrt_small] Epoch 13/20 - Loss: 0.0444
[esrt_small] Epoch 14/20 - Loss: 0.0443
[esrt_small] Epoch 15/20 - Loss: 0.0439
[esrt_small] Checkpoint saved: /kaggle/working/checkpoints/esrt_small_epoch15.pth
[esrt_small] Epoch 16/20 - Loss: 0.0433
[esrt_small] Epoch 1